## Importing the libraries

In [2]:
import pandas as pd

## Load datasets

In [5]:
math_df = pd.read_csv('student-mat.csv', sep=';')
por_df = pd.read_csv('student-por.csv', sep=';')
print("Math shape:", math_df.shape)
print("Portugese shpae:", por_df.shape)

Math shape: (395, 33)
Portugese shpae: (649, 33)


## Datasets cleaning

In [6]:
### Adding Subject column
math_df['subject'] = 'math'
por_df['subject'] = 'portugese'

In [9]:
### Merging the two dataframes
combined_df = pd.concat([math_df, por_df], axis=0)

### Reseting the index
combined_df.reset_index(drop=True, inplace=True)

print("Combined shape:", combined_df.shape)
combined_df.head()

Combined shape: (1044, 34)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,subject
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,math
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,math
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,math
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,math
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,math


## Creating the risk label

In [11]:
df = combined_df.copy()

### Regression target
y_reg = df["G3"]

### Classification target: risk level from G3
def risk_label(g3):
    if g3 >= 15:
        return "Low"
    elif g3 >= 10:
        return "Medium"
    else:
        return "High"

df["risk_level"] = df["G3"].apply(risk_label)

print(df["risk_level"].value_counts())

risk_level
Medium    610
High      230
Low       204
Name: count, dtype: int64


## Train Test Split

In [60]:
from sklearn.model_selection import train_test_split

### Features
X = df.drop(["G3", "risk_level"], axis=1)

### Targets
y_reg = df["G3"]
y_cls = df["risk_level"]

### Train-Test split
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(X,
    y_reg,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (835, 33)
Test set size: (209, 33)


In [61]:
print(y_cls_train.value_counts(normalize=True))
print(y_cls_test.value_counts(normalize=True))

risk_level
Medium    0.584431
High      0.220359
Low       0.195210
Name: proportion, dtype: float64
risk_level
Medium    0.583732
High      0.220096
Low       0.196172
Name: proportion, dtype: float64


## Data Preprocessing

In [62]:
### Identifying categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns

In [63]:
print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))

Categorical columns: 18
Numerical columns: 15


In [64]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

### Encoding categorical columns
preprocessor = ColumnTransformer(transformers = [("cat", OneHotEncoder(handle_unknown = "ignore"), categorical_cols), ("num", StandardScaler(), numerical_cols)], remainder="drop")

In [65]:
print(preprocessor)

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic', 'subject'],
      dtype='object')),
                                ('num', StandardScaler(),
                                 Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2'],
      dtype='object'))])


In [66]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [67]:
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (835, 60)
Processed test shape: (209, 60)


## Linear Regression

In [68]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

### Pipeline
linreg_pipeline = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])

### Train
linreg_pipeline.fit(X_train, y_reg_train)

### Predict
y_pred_lin = linreg_pipeline.predict(X_test)

### Evaluate
mae_lin = mean_absolute_error(y_reg_test, y_pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_reg_test, y_pred_lin))
r2_lin = r2_score(y_reg_test, y_pred_lin)

In [69]:
print("Linear Regression Results:")
print("MAE:", mae_lin)
print("RMSE:", rmse_lin)
print("R2:", r2_lin)

Linear Regression Results:
MAE: 1.0555926838239287
RMSE: 1.6301742781402921
R2: 0.8194608947189415


## Random Forest Regression

In [70]:
from sklearn.ensemble import RandomForestRegressor

rf_reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators = 200,
        random_state = 42,
        n_jobs = -1
    ))
])

rf_reg_pipeline.fit(X_train, y_reg_train)

y_pred_rf = rf_reg_pipeline.predict(X_test)

mae_rf = mean_absolute_error(y_reg_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_reg_test, y_pred_rf))
r2_rf = r2_score(y_reg_test, y_pred_rf)

In [71]:
print("Random Forest Regressor Results:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

Random Forest Regressor Results:
MAE: 0.9157416267942583
RMSE: 1.295437521922899
R2: 0.8859917820171455


## Logistic Regression (Baseline Model)

In [72]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logreg_cls_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

logreg_cls_pipeline.fit(X_train, y_cls_train)

y_pred_log = logreg_cls_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_cls_test, y_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_cls_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_cls_test, y_pred_log))

Logistic Regression Accuracy: 0.8516746411483254

Confusion Matrix:
 [[ 38   0   8]
 [  0  31  10]
 [  6   7 109]]

Classification Report:
               precision    recall  f1-score   support

        High       0.86      0.83      0.84        46
         Low       0.82      0.76      0.78        41
      Medium       0.86      0.89      0.88       122

    accuracy                           0.85       209
   macro avg       0.85      0.83      0.83       209
weighted avg       0.85      0.85      0.85       209



## Random Forest Classification

In [73]:
from sklearn.ensemble import RandomForestClassifier

rf_cls_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"  # helps because Medium is the largest class
    ))
])

rf_cls_pipeline.fit(X_train, y_cls_train)

y_pred_rf_cls = rf_cls_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_cls_test, y_pred_rf_cls))
print("\nConfusion Matrix:\n", confusion_matrix(y_cls_test, y_pred_rf_cls))
print("\nClassification Report:\n", classification_report(y_cls_test, y_pred_rf_cls))

Random Forest Accuracy: 0.8325358851674641

Confusion Matrix:
 [[ 29   0  17]
 [  0  35   6]
 [  5   7 110]]

Classification Report:
               precision    recall  f1-score   support

        High       0.85      0.63      0.72        46
         Low       0.83      0.85      0.84        41
      Medium       0.83      0.90      0.86       122

    accuracy                           0.83       209
   macro avg       0.84      0.80      0.81       209
weighted avg       0.83      0.83      0.83       209



## Trained models

In [76]:
import joblib

joblib.dump(rf_reg_pipeline, "rf_reg_pipeline.pkl")
joblib.dump(logreg_cls_pipeline, "logreg_cls_pipeline.pkl")

Saved models!


## Generalization Check for Regression

In [74]:
### Checking overfitting for Linear Regression 

### Training predictions
y_pred_train_lin = linreg_pipeline.predict(X_train)

### Training metrics
r2_train_lin = r2_score(y_reg_train, y_pred_train_lin)
r2_test_lin = r2_lin  # already computed earlier

print("Linear Regression Train R2:", r2_train_lin)
print("Linear Regression Test R2:", r2_test_lin)

### Checking overfitting for Random Forest Regression 

### Training predictions
y_pred_train_rf = rf_reg_pipeline.predict(X_train)

### Training metrics
r2_train_rf = r2_score(y_reg_train, y_pred_train_rf)
r2_test_rf = r2_rf  # already computed earlier

print("Random Forest Train R2:", r2_train_rf)
print("Random Forest Test R2:", r2_test_rf)

Linear Regression Train R2: 0.8501935995208422
Linear Regression Test R2: 0.8194608947189415
Random Forest Train R2: 0.9777164144265635
Random Forest Test R2: 0.8859917820171455


## Generalization Check for Classification

In [75]:
from sklearn.metrics import f1_score

print("RF Train Accuracy:", rf_cls_pipeline.score(X_train, y_cls_train))
print("RF Test Accuracy:", rf_cls_pipeline.score(X_test, y_cls_test))

print("RF Train F1 (macro):", f1_score(y_cls_train, rf_cls_pipeline.predict(X_train), average="macro"))
print("RF Test F1 (macro):", f1_score(y_cls_test, y_pred_rf_cls, average="macro"))

RF Train Accuracy: 1.0
RF Test Accuracy: 0.8325358851674641
RF Train F1 (macro): 1.0
RF Test F1 (macro): 0.8103728640050397
